In [1]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv('../data/processed/investments_VC_cleaned.csv')
df_known = df[df['status'] != 'unknown'].copy()
df_known['is_success'] = df_known['status'].isin(['operating', 'acquired']).astype(int)
print('Loaded: ' + str(df.shape))

Loaded: (49436, 39)


In [2]:
sector_kpi = df_known.groupby('market').agg(
    total_startups=('is_success', 'count'),
    successful=('is_success', 'sum'),
    success_rate=('is_success', 'mean'),
    avg_funding=('funding_total_usd', 'mean'),
    median_funding=('funding_total_usd', 'median'),
    avg_rounds=('funding_rounds', 'mean')
).reset_index()

sector_kpi['success_rate_pct'] = (sector_kpi['success_rate'] * 100).round(2)
sector_kpi = sector_kpi[sector_kpi['total_startups'] >= 50]
sector_kpi = sector_kpi.sort_values('success_rate_pct', ascending=False)

sector_kpi.to_csv('../data/processed/kpi_sector_summary.csv', index=False)
print('Sector KPI: ' + str(sector_kpi.shape))
print(sector_kpi.head().to_string())

Sector KPI: (102, 8)
                 market  total_startups  successful  success_rate   avg_funding  median_funding  avg_rounds  success_rate_pct
362  Internet Of Things              72          72           1.0  2.821220e+06        858812.5    1.847222             100.0
60              Bitcoin              54          54           1.0  3.270747e+06       1187500.0    1.537037             100.0
128           Consumers              51          51           1.0  3.488946e+06        800000.0    1.450980             100.0
257  Finance Technology              64          64           1.0  4.834996e+06       1822979.0    1.718750             100.0
518     Pharmaceuticals              81          81           1.0  7.073896e+06       4000000.0    1.444444             100.0


In [3]:
country_kpi = df_known.groupby('country_code').agg(
    total_startups=('is_success', 'count'),
    successful=('is_success', 'sum'),
    success_rate=('is_success', 'mean'),
    avg_funding=('funding_total_usd', 'mean'),
    total_funding=('funding_total_usd', 'sum')
).reset_index()

country_kpi['success_rate_pct'] = (country_kpi['success_rate'] * 100).round(2)
country_kpi = country_kpi[country_kpi['total_startups'] >= 30]

country_kpi.to_csv('../data/processed/kpi_country_summary.csv', index=False)
print('Country KPI: ' + str(country_kpi.shape))

Country KPI: (50, 7)


In [4]:
round_cols = ['seed','angel','round_a','round_b','round_c','round_d','round_e','round_f']
funnel_data = []
for col in round_cols:
    total = (df[col] > 0).sum()
    success = df_known[df_known[col] > 0]['is_success'].mean()
    funnel_data.append({'stage': col, 'count': total, 'success_rate': round(success*100, 2)})

funnel_df = pd.DataFrame(funnel_data)
funnel_df.to_csv('../data/processed/kpi_funding_funnel.csv', index=False)
print(funnel_df.to_string())

     stage  count  success_rate
0     seed  13839         94.43
1    angel   3129         91.56
2  round_a   9003         94.90
3  round_b   5447         95.41
4  round_c   2837         96.51
5  round_d   1288         97.13
6  round_e    516         96.71
7  round_f    172         96.51


In [5]:
tableau_cols = ['name','market','status','is_success','country_code',
                'founded_year','funding_rounds','funding_total_usd',
                'seed','venture','round_a','round_b','round_c']

df_tableau = df_known[tableau_cols].copy()
df_tableau.to_csv('../data/processed/tableau_ready.csv', index=False)
print('Tableau-ready: ' + str(df_tableau.shape))
print('All KPI files saved to data/processed/')

Tableau-ready: (49436, 13)
All KPI files saved to data/processed/
